In [1]:
# EnigmaRoom - AI-Powered Escape Challenge
# Version 1: Interactive AI Escape Room

print("=" * 55)
print("       🔐 WELCOME TO ENIGMAROOM 🔐")
print("       AI-POWERED ESCAPE CHALLENGE")
print("=" * 55)

print("""
You wake up inside a mysterious library.

The exit door is locked with a four-digit code.

You can see:
1. A writing desk
2. A bookshelf
3. An ancient portrait
4. A locked drawer
5. An exit door

Your mission:
Explore the room, discover hidden clues,
solve the puzzles and escape!

Good luck! 🗝️
""")

       🔐 WELCOME TO ENIGMAROOM 🔐
       AI-POWERED ESCAPE CHALLENGE

You wake up inside a mysterious library.

The exit door is locked with a four-digit code.

You can see:
1. A writing desk
2. A bookshelf
3. An ancient portrait
4. A locked drawer
5. An exit door

Your mission:
Explore the room, discover hidden clues,
solve the puzzles and escape!

Good luck! 🗝️



In [2]:
# Install required libraries for EnigmaRoom
!pip install -q google-genai
!pip install -q streamlit

print("Libraries installed successfully!")

Libraries installed successfully!


In [3]:
import os
from getpass import getpass
from google import genai

# Securely enter your Gemini API key
GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini AI configuration completed!")

Enter your Gemini API Key: ··········
Gemini AI configuration completed!


In [ ]:
# Test Gemini AI connection

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="You are the Game Master of an escape room. Welcome the player to a mysterious library in two sentences."
)

print("Gemini AI Response:")
print(response.text)

In [5]:
# Test Gemini AI connection with the updated model

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="""
    You are the Game Master of an escape room.
    Welcome the player to a mysterious library.
    Make the introduction exciting and mysterious.
    """
)

print("ENIGMAROOM - GEMINI AI")
print("=" * 50)
print(response.text)

KeyboardInterrupt: 

In [6]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say: Welcome to EnigmaRoom!"
)

print(response.text)

KeyboardInterrupt: 

In [7]:
from google.genai import types

# Set a 20-second timeout
client = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(timeout=20000)
)

try:
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="Say hello in one sentence."
    )
    print("Gemini connected successfully!")
    print(response.text)

except Exception as e:
    print("Connection issue:", type(e).__name__)
    print(str(e)[:500])

Connection issue: ServerError
503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


In [8]:
!pip install -q gradio

In [9]:
import gradio as gr

SYSTEM_PROMPT = """
You are the Game Master of EnigmaRoom,
an AI-powered escape room game.

The player is trapped inside a mysterious library.

There are five objects:
1. Writing desk
2. Bookshelf
3. Ancient portrait
4. Locked drawer
5. Exit door

The player must discover three clues and solve puzzles.

The final escape code is 4317.

Guide the player through the story.
Respond naturally to their actions.
Remember the conversation.
Give clues gradually.
Do not allow escape without the correct code.
"""

def play_game(message, history):
    try:
        conversation = SYSTEM_PROMPT + "\n\nPrevious conversation:\n"

        for user_msg, assistant_msg in history:
            conversation += f"Player: {user_msg}\nGame Master: {assistant_msg}\n"

        conversation += f"\nPlayer: {message}\nGame Master:"

        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=conversation
        )

        return response.text

    except Exception as e:
        return (
            "The library lights flicker... The Game Master "
            "is temporarily unavailable. Please try again shortly. "
            f"Technical detail: {type(e).__name__}"
        )

demo = gr.ChatInterface(
    fn=play_game,
    title="🔐 EnigmaRoom – AI Escape Challenge",
    description=(
        "You are trapped inside a mysterious library. "
        "Explore objects, discover clues and escape!"
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b0f67a99acf821728.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
import gradio as gr

# ==========================================
# ENIGMAROOM - VERSION 1
# AI ESCAPE ROOM WITH LOCAL GAME FALLBACK
# ==========================================

game = {
    "key": False,
    "uv_light": False,
    "note": False,
    "portrait_clue": False,
    "drawer_open": False,
    "escaped": False
}

def reset_game():
    for item in game:
        game[item] = False

def local_game(message, history):

    msg = message.lower().strip()

    # Restart
    if "restart" in msg:
        reset_game()
        return "🔄 The library resets. Your adventure begins again!"

    # Bookshelf
    if "bookshelf" in msg or "books" in msg:
        if not game["key"]:
            game["key"] = True
            return (
                "📚 You carefully inspect the bookshelf. "
                "Behind an ancient book, you discover a BRASS KEY! "
                "You add it to your inventory."
            )
        return "You have already searched the bookshelf."

    # Locked drawer
    if "locked drawer" in msg or "drawer" in msg:
        if not game["key"]:
            return "🔒 The drawer is locked. Find a key before opening it."

        if not game["drawer_open"]:
            game["drawer_open"] = True
            game["uv_light"] = True
            return (
                "🗝️ Your brass key opens the locked drawer! "
                "Inside, you discover a UV FLASHLIGHT. "
                "You collect it."
            )

        return "The drawer is already open."

    # Writing desk
    if "desk" in msg or "note" in msg:
        if not game["drawer_open"]:
            return "🪑 You notice a locked compartment near the desk. Open the locked drawer first."

        if not game["note"]:
            game["note"] = True
            return (
                "📜 You examine the writing desk and discover "
                "a hidden note. It reveals the first two digits: 43."
            )

        return "You have already discovered the note."

    # Portrait
    if "portrait" in msg or "painting" in msg:
        if not game["uv_light"]:
            return "🖼️ The portrait contains mysterious markings. You need a special light."

        if not game["portrait_clue"]:
            game["portrait_clue"] = True
            return (
                "🔦 You shine the UV flashlight on the portrait. "
                "Hidden writing appears: THE FINAL TWO DIGITS ARE 17."
            )

        return "The portrait's hidden message has already been discovered."

    # Inventory
    if "inventory" in msg or "items" in msg:
        items = []

        if game["key"]:
            items.append("Brass Key")

        if game["uv_light"]:
            items.append("UV Flashlight")

        if game["note"]:
            items.append("Secret Note")

        return "🎒 YOUR INVENTORY:\n" + (
            "\n".join("• " + item for item in items)
            if items else "Empty"
        )

    # Exit door
    if "door" in msg or "exit" in msg or any(x in msg for x in ["4317", "1234"]):

        if not game["note"] or not game["portrait_clue"]:
            return (
                "🚪 The exit remains locked! "
                "You must discover all the clues before escaping."
            )

        if "4317" in msg:
            game["escaped"] = True
            return (
                "🎉 CONGRATULATIONS! 🎉\n\n"
                "The keypad turns green. The door opens!\n"
                "You have successfully escaped ENIGMAROOM!"
            )

        return "🔐 Enter the four-digit code to unlock the exit."

    return (
        "🔍 I don't recognise that action.\n\n"
        "Try:\n"
        "• Examine bookshelf\n"
        "• Open locked drawer\n"
        "• Examine desk\n"
        "• Inspect portrait\n"
        "• Check inventory\n"
        "• Unlock exit door"
    )


# Create the game interface
demo_v1 = gr.ChatInterface(
    fn=local_game,
    title="🔐 EnigmaRoom – AI Escape Challenge",
    description="""
    Welcome to the mysterious library!

    Find the hidden key, discover the secret clues
    and unlock the four-digit exit code.

    Your mission: ESCAPE!
    """,
    examples=[
        "Examine the bookshelf",
        "Open the locked drawer",
        "Examine the desk",
        "Inspect the portrait",
        "Check inventory"
    ]
)

demo_v1.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2b2dcdb9655cf7b770.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
